# March Madness — poprawny time split bez leakage

Ten notebook:
- wczytuje `fulldataset.csv`
- **nie duplikuje meczów** i **nie zamienia** ich na dwa rekordy
- zostawia strukturę **1 wiersz = 1 realny mecz**
- robi **split po czasie / po sezonach**
- zapisuje gotowe pliki CSV dla:
  - finalnego holdoutu (`train` / `test`)
  - rolling / expanding CV (`train` / `valid`)
- tworzy manifest ze wszystkimi splitami

## Ważne założenie
Notebook **nie rozwiązuje leakage samym splitem**, jeśli w `fulldataset.csv` masz już kolumny policzone z przyszłości albo statystyki z **tego samego meczu**.

Przykłady kolumn, których **nie wolno** używać do predykcji przedmeczowej:
- `WScore`, `LScore`
- `NumOT`
- boxscore z tego meczu: `WFGM`, `LFGM`, `WAst`, `LTO`, itd.
- jakiekolwiek kolumny opisujące wynik bieżącego meczu

Do modelu wolno brać tylko:
- informacje znane **przed rozpoczęciem meczu**
- rolling features liczone wyłącznie z wcześniejszych spotkań
- rankingi dostępne przed meczem
- seed tylko wtedy, gdy modelujesz mecze turniejowe po Selection Sunday


In [ ]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

CONFIG = {
    "input_path": "fulldataset.csv",
    "output_dir": "time_splits",
    "season_col": "Season",
    "day_col_candidates": ["DayNum", "GameDayNum", "RankingDayNum"],
    "date_col_candidates": ["GameDate", "Date", "game_date", "date"],
    "game_id_col_candidates": ["GameID", "game_id", "ID", "MatchID", "match_id", "meczid", "MeczID"],
    "holdout_test_seasons": 1,     # ile ostatnich sezonów odkładamy jako finalny test
    "min_train_seasons": 8,        # minimum sezonów w treningu dla CV
    "val_seasons": 1,              # ile kolejnych sezonów w walidacji
    "step": 1,                     # o ile sezonów przesuwamy okno
    "save_index": False,
}

INPUT_PATH = Path(CONFIG["input_path"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Nie znaleziono pliku: {INPUT_PATH.resolve()}\\n"
        "Umieść `fulldataset.csv` w tym samym folderze co notebook "
        "albo zmień CONFIG['input_path']."
    )

df = pd.read_csv(INPUT_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None

def infer_columns(df, config):
    season_col = config["season_col"]
    if season_col not in df.columns:
        raise ValueError(
            f"Brakuje kolumny `{season_col}`. "
            f"Dostępne kolumny: {list(df.columns)[:30]}..."
        )

    day_col = first_existing(df.columns, config["day_col_candidates"])
    date_col = first_existing(df.columns, config["date_col_candidates"])
    game_id_col = first_existing(df.columns, config["game_id_col_candidates"])

    return season_col, day_col, date_col, game_id_col

def prepare_chronology(df, season_col, day_col=None, date_col=None, game_id_col=None):
    out = df.copy()

    out[season_col] = pd.to_numeric(out[season_col], errors="raise").astype(int)

    if date_col is not None:
        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    if day_col is None and date_col is None:
        warnings.warn(
            "Nie znaleziono ani DayNum, ani Date/GameDate. "
            "Split będzie robiony tylko po kolumnie Season."
        )

    out["_row_order"] = np.arange(len(out))

    sort_cols = [season_col]
    if day_col is not None:
        sort_cols.append(day_col)
    elif date_col is not None:
        if out[date_col].isna().all():
            raise ValueError(
                f"Kolumna `{date_col}` istnieje, ale po parsowaniu wszystkie daty są NaT."
            )
        sort_cols.append(date_col)

    if game_id_col is not None:
        sort_cols.append(game_id_col)

    sort_cols.append("_row_order")
    out = out.sort_values(sort_cols).reset_index(drop=True)

    return out, sort_cols

def safe_basic_eda(df, season_col, day_col=None, date_col=None, game_id_col=None):
    print("===== BASIC SAFE EDA (bez patrzenia w target/finalny wynik) =====")
    print("Liczba wierszy:", len(df))
    print("Liczba kolumn:", df.shape[1])
    print("\\nTypy kolumn:")
    display(df.dtypes.to_frame("dtype").head(50))

    print("\\nBraki danych (%):")
    missing = (df.isna().mean().sort_values(ascending=False) * 100).round(2)
    display(missing[missing > 0].to_frame("missing_pct").head(50))

    print("\\nLiczba meczów per sezon:")
    season_counts = df.groupby(season_col).size().rename("n_games").reset_index()
    display(season_counts)

    if day_col is not None:
        print("\\nZakres DayNum per sezon:")
        day_summary = df.groupby(season_col)[day_col].agg(["min", "max", "nunique"]).reset_index()
        display(day_summary)

    if date_col is not None:
        print("\\nZakres dat per sezon:")
        date_summary = df.groupby(season_col)[date_col].agg(["min", "max"]).reset_index()
        display(date_summary)

    if game_id_col is not None:
        dupes = df[game_id_col].duplicated().sum()
        print(f"\\nDuplikaty po `{game_id_col}`:", dupes)
        if dupes > 0:
            warnings.warn(
                f"Masz {dupes} duplikatów po `{game_id_col}`. "
                "Jeśli 1 wiersz ma oznaczać 1 mecz, trzeba to sprawdzić przed trenowaniem."
            )

def find_potential_leakage_columns(columns):
    suspicious_tokens = [
        "score", "result", "winner", "loser", "margin", "mov", "ot",
        "fgm", "fga", "fgm3", "fga3", "ftm", "fta",
        "reb", "or", "dr", "ast", "to", "stl", "blk", "pf",
        "wscore", "lscore", "wloc", "numot"
    ]
    found = []
    for col in columns:
        c = str(col).lower()
        if any(tok in c for tok in suspicious_tokens):
            found.append(col)
    return sorted(set(found))

season_col, day_col, date_col, game_id_col = infer_columns(df, CONFIG)
df, sort_cols = prepare_chronology(
    df=df,
    season_col=season_col,
    day_col=day_col,
    date_col=date_col,
    game_id_col=game_id_col,
)

print("Użyte kolumny chronologiczne:")
print({
    "season_col": season_col,
    "day_col": day_col,
    "date_col": date_col,
    "game_id_col": game_id_col,
    "sort_cols": sort_cols,
})

safe_basic_eda(df, season_col, day_col, date_col, game_id_col)

suspicious = find_potential_leakage_columns(df.columns)
print("\\nPotencjalnie podejrzane kolumny (do audytu przed modelowaniem):")
print(suspicious[:100])


In [ ]:
def summarize_frame(frame, name, role, path, season_col, day_col=None, date_col=None):
    row = {
        "split_name": name,
        "role": role,
        "path": str(path),
        "rows": len(frame),
        "first_season": int(frame[season_col].min()) if len(frame) else np.nan,
        "last_season": int(frame[season_col].max()) if len(frame) else np.nan,
    }
    if day_col is not None and len(frame):
        row["min_daynum"] = int(frame[day_col].min())
        row["max_daynum"] = int(frame[day_col].max())
    else:
        row["min_daynum"] = np.nan
        row["max_daynum"] = np.nan

    if date_col is not None and len(frame):
        row["min_date"] = frame[date_col].min()
        row["max_date"] = frame[date_col].max()
    else:
        row["min_date"] = pd.NaT
        row["max_date"] = pd.NaT
    return row

unique_seasons = sorted(df[season_col].dropna().unique().tolist())
print("Sezony:", unique_seasons)

if len(unique_seasons) < (CONFIG["min_train_seasons"] + CONFIG["val_seasons"] + CONFIG["holdout_test_seasons"]):
    raise ValueError(
        "Za mało sezonów do zadanych parametrów splitu. "
        "Zmniejsz `min_train_seasons` albo `holdout_test_seasons`."
    )

holdout_test_seasons = unique_seasons[-CONFIG["holdout_test_seasons"]:]
holdout_train_seasons = unique_seasons[:-CONFIG["holdout_test_seasons"]]

train_holdout = df[df[season_col].isin(holdout_train_seasons)].copy()
test_holdout = df[df[season_col].isin(holdout_test_seasons)].copy()

holdout_dir = OUTPUT_DIR / "final_holdout"
holdout_dir.mkdir(parents=True, exist_ok=True)

train_holdout_path = holdout_dir / "train.csv"
test_holdout_path = holdout_dir / "test.csv"

train_holdout.to_csv(train_holdout_path, index=CONFIG["save_index"])
test_holdout.to_csv(test_holdout_path, index=CONFIG["save_index"])

manifest_rows = []
manifest_rows.append(
    summarize_frame(train_holdout, "final_holdout", "train", train_holdout_path, season_col, day_col, date_col)
)
manifest_rows.append(
    summarize_frame(test_holdout, "final_holdout", "test", test_holdout_path, season_col, day_col, date_col)
)

print("Zapisano finalny holdout:")
print(train_holdout_path.resolve())
print(test_holdout_path.resolve())

print("\\nHoldout train sezony:", holdout_train_seasons[0], "->", holdout_train_seasons[-1])
print("Holdout test sezony:", holdout_test_seasons)
print("Holdout shapes:", train_holdout.shape, test_holdout.shape)


In [ ]:
cv_dir = OUTPUT_DIR / "rolling_cv"
cv_dir.mkdir(parents=True, exist_ok=True)

base_cv_seasons = holdout_train_seasons[:]  # CV tylko na historii, bez finalnego holdoutu
min_train = CONFIG["min_train_seasons"]
val_seasons = CONFIG["val_seasons"]
step = CONFIG["step"]

fold = 1
for train_end in range(min_train, len(base_cv_seasons) - val_seasons + 1, step):
    train_seasons_fold = base_cv_seasons[:train_end]
    valid_seasons_fold = base_cv_seasons[train_end:train_end + val_seasons]

    if len(valid_seasons_fold) < val_seasons:
        break

    train_fold_df = df[df[season_col].isin(train_seasons_fold)].copy()
    valid_fold_df = df[df[season_col].isin(valid_seasons_fold)].copy()

    fold_name = f"fold_{fold:02d}"
    fold_dir = cv_dir / fold_name
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_path = fold_dir / "train.csv"
    valid_path = fold_dir / "valid.csv"

    train_fold_df.to_csv(train_path, index=CONFIG["save_index"])
    valid_fold_df.to_csv(valid_path, index=CONFIG["save_index"])

    manifest_rows.append(
        summarize_frame(train_fold_df, fold_name, "train", train_path, season_col, day_col, date_col)
    )
    manifest_rows.append(
        summarize_frame(valid_fold_df, fold_name, "valid", valid_path, season_col, day_col, date_col)
    )

    print(
        f"{fold_name}: train {train_seasons_fold[0]}->{train_seasons_fold[-1]} | "
        f"valid {valid_seasons_fold[0]}->{valid_seasons_fold[-1]} | "
        f"rows {train_fold_df.shape[0]} / {valid_fold_df.shape[0]}"
    )
    fold += 1

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_DIR / "split_manifest.csv"
manifest.to_csv(manifest_path, index=False)

config_path = OUTPUT_DIR / "split_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

print("\\nZapisano manifest:")
print(manifest_path.resolve())
display(manifest.head(20))


# Jak dalej pracować, żeby nie zrobić data leakage

## 1. Co wolno zrobić **przed splitem**
To jest bezpieczne:
- sprawdzenie schematu kolumn
- liczba meczów per sezon
- braki danych
- duplikaty
- zakres dat / DayNum
- kontrola jakości `GameID`

To **nie jest jeszcze** modelowe EDA, więc jest OK.

---

## 2. Kiedy robić feature engineering
Są dwa bezpieczne warianty.

### Wariant A — najbezpieczniejszy praktycznie
1. bierzesz **raw mecze**
2. robisz **time split**
3. **w każdym foldzie osobno** liczysz:
   - rolling averages
   - ELO
   - win rate do danego dnia
   - stats z ostatnich N meczów
   - strength of schedule
4. fitujesz imputery / scalery / encodery **tylko na train**
5. transformujesz valid / test tym, co nauczyło się na train

To jest najbardziej restrykcyjne i najtrudniej tu o błąd.

### Wariant B — też poprawny, ale tylko jeśli featury są ściśle kauzalne
Możesz policzyć feature engineering **na całej historii naraz**, ale tylko wtedy, gdy:
- idziesz chronologicznie mecz po meczu
- feature dla meczu z dnia `t` korzysta wyłącznie z meczów `< t`
- nigdzie nie używasz pełnego sezonu ani przyszłych wyników

Przykład poprawny:
- win rate drużyny liczony do dnia meczu
- ELO po poprzednim meczu
- średnia punktów z 5 poprzednich spotkań

Przykład niepoprawny:
- końcowy win rate z całego sezonu
- seed lub ranking, który pojawił się dopiero później
- średnia punktów liczona z meczem bieżącym włącznie albo z przyszłymi meczami

---

## 3. Kiedy robić „prawdziwe” EDA
Po splicie.

Najlepiej tak:
- pełne EDA rozkładów i korelacji robisz na `train`
- `valid` służy do strojenia
- `test` zostawiasz na sam koniec i nie podejmujesz na jego podstawie decyzji projektowych

Na `test` możesz obejrzeć tylko końcowe metryki i ewentualnie prostą diagnostykę po zamrożeniu pipeline'u.

---

## 4. Bardzo ważne przy Twoim układzie 1 wiersz = 1 mecz
Jeżeli zostawiasz rekord jako:
- `WTeamID`
- `LTeamID`

to taki dataframe **nie jest jeszcze gotowym datasetem do klasyfikacji binarnej**, bo zwycięzca jest zawsze po stronie `WTeamID`.

Masz potem 2 sensowne drogi:

### Droga 1 — model rangowy / ratingowy
Nie robisz klasycznego `target` 0/1 na surowym układzie.
Zamiast tego liczysz np.:
- ELO
- Glicko
- Bradley-Terry
- inne ratingi drużyn

a potem z różnicy ratingów wyznaczasz prawdopodobieństwo wygranej nowego meczu.

### Droga 2 — jeden rekord = jeden mecz, ale z kanonicznym uporządkowaniem
Dopiero **po splicie**, wewnątrz foldów, tworzysz wersję modelową:
- `Team1 = min(TeamID_A, TeamID_B)`
- `Team2 = max(TeamID_A, TeamID_B)`
- `target = 1`, jeśli wygrał `Team1`, w przeciwnym razie `0`

To dalej jest **1 mecz = 1 rekord**, tylko już w formacie nadającym się do modelu klasyfikacyjnego.

To nie jest to samo co dublowanie meczów.  
Nie robisz dwóch wierszy, tylko jeden.

---

## 5. Co robić po tym notebooku
Dobra kolejność prac:

1. **uruchom notebook i wygeneruj splity**
2. sprawdź `split_manifest.csv`
3. wybierz jedną strategię modelowania:
   - rating / ELO
   - albo klasyfikator na wersji kanonicznej 1-row-per-game
4. dopiero potem buduj features
5. scalery / imputery / encoding fituj wyłącznie na train
6. model wybieraj po CV
7. finalny wynik oceniaj dopiero na `final_holdout/test.csv`

---

## 6. Minimalna praktyczna rekomendacja
Na Twoim miejscu robiłbym to tak:

- teraz: **raw split po sezonach** tym notebookiem
- potem: osobny notebook **feature engineering per fold**
- potem: osobny notebook **training + CV + logloss**

To daje najczystszy pipeline i najłatwiej kontrolować leakage.


In [ ]:
print("Folder z wynikami:", OUTPUT_DIR.resolve())
print("Pliki końcowe:")
for p in sorted(OUTPUT_DIR.rglob("*.csv")):
    print("-", p)
